# 04 · Status da plataforma SCCON
Lê o relatório de status da plataforma, guarda o status mais recente de cada
alerta e confere a cobertura em relação à planilha da CMF.

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# Caminhos do projeto (relativos à pasta notebooks/)
bronze_dir = Path("../dados/bronze")
silver_dir = Path("../dados/prata")
sccon_file = bronze_dir / "sccon_alertas_2026-11-21.csv"  # ajuste se não renomear

In [ ]:
# Lê o extrato da SCCON (codificação Latin-1; separador detectado automaticamente)
df_sccon = pd.read_csv(sccon_file, encoding="ISO-8859-1", sep=None, engine="python")
print(df_sccon.shape)
print(list(df_sccon.columns))

# Colunas com poucas categorias: mostra os valores de cada uma
for column in df_sccon.columns:
    n_values = df_sccon[column].nunique()
    if n_values <= 20:
        print(f"\n=== {column} ({n_values} valores)")
        print(df_sccon[column].value_counts(dropna=False).to_string())

In [ ]:
# Colunas úteis, com nomes padronizados (sem nome de usuário e, por ora, sem geometria)
sccon_columns = {
    "ID Alert": "id_alerta",
    "Data de Inserção": "data_insercao",
    "Status": "status_sccon",
    "Observação": "unidade_responsavel",
    "classe": "tipo_alerta",
    "mun_princ": "municipio",
    "dat_antes": "data_imagem_antes",
    "dat_depois": "data_imagem_depois",
    "area_ha": "area_ha",
}
df_sccon = df_sccon[list(sccon_columns)].rename(columns=sccon_columns)
df_sccon["id_alerta"] = df_sccon["id_alerta"].astype("int64")
for column in ["data_imagem_antes", "data_imagem_depois"]:
    df_sccon[column] = pd.to_datetime(df_sccon[column], format="%Y-%m-%d", errors="coerce")

# Cada carga de status gera uma linha: fica o registro mais recente de cada alerta
df_sccon["data_insercao"] = pd.to_datetime(df_sccon["data_insercao"], dayfirst=True)
n_records = len(df_sccon)
df_sccon = (df_sccon.sort_values("data_insercao")
                    .drop_duplicates("id_alerta", keep="last"))
print(n_records, "registros de status |", len(df_sccon), "alertas únicos")
print(df_sccon["status_sccon"].value_counts().to_string())

In [ ]:
# Os dois conjuntos de alertas e o período que cada um cobre (pela data da imagem)
df_cmf_alerts = pd.read_parquet(silver_dir / "alertas.parquet")
sccon_ids = set(df_sccon["id_alerta"])
cmf_ids = set(df_cmf_alerts["id_alerta"])

print("Nas duas:", len(sccon_ids & cmf_ids),
      "| Só na SCCON:", len(sccon_ids - cmf_ids),
      "| Só na CMF:", len(cmf_ids - sccon_ids))
print("SCCON:", df_sccon["data_imagem_depois"].min().date(), "a", df_sccon["data_imagem_depois"].max().date())
print("CMF:  ", df_cmf_alerts["data_imagem_depois"].min().date(), "a", df_cmf_alerts["data_imagem_depois"].max().date())

In [ ]:
# A coluna Observação é texto livre: fica só quando é uma unidade (BPMA ou GAEF-...)
is_unit = df_sccon["unidade_responsavel"].str.match(r"^(BPMA|GAEF-\w+)$", na=False)
df_sccon["unidade_responsavel"] = df_sccon["unidade_responsavel"].where(is_unit)

# Status mais recente de cada alerta na plataforma, para a camada ouro
status_columns = ["id_alerta", "status_sccon", "unidade_responsavel", "data_insercao"]
df_sccon[status_columns].to_parquet(silver_dir / "status_sccon.parquet", index=False)
print(len(df_sccon), "status salvos")